# Multilingual Wikipedia: Token-Level Coherence Decay

Cross-linguistic test of the universal coherence decay exponent across **6 non-Indo-European language families**:

| Language | Family | Typology | Articles |
|----------|--------|----------|----------|
| Chinese | Sino-Tibetan | isolating, SVO | 60 |
| Japanese | Japonic | agglutinative, SOV | 60 |
| Korean | Koreanic | agglutinative, SOV | 60 |
| Turkish | Turkic | agglutinative, SOV | 60 |
| Arabic | Afro-Asiatic | root-pattern, VSO/SVO | 60 |
| Finnish | Uralic | agglutinative, rich case | 60 |

**Prior results** (Indo-European + English spoken):
- English written (RAID): α = -0.75
- English spoken (Buckeye): α = -0.73
- French spoken (Oral Narrative): α = -0.69
- Anderson & Schooler (1991): α = -0.77

**Method**: Identical language-agnostic token-reveal pipeline — no sentence parsing, no language-specific processing. Mistral-7B as probe (trained on multilingual data).

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print('Imports OK')

In [ ]:
# === Configuration ===
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

LANGUAGES = {
    'zh': {'name': 'Chinese',  'family': 'Sino-Tibetan',  'color': '#e6194b'},
    'ja': {'name': 'Japanese', 'family': 'Japonic',       'color': '#f58231'},
    'ko': {'name': 'Korean',   'family': 'Koreanic',      'color': '#ffe119'},
    'tr': {'name': 'Turkish',  'family': 'Turkic',        'color': '#3cb44b'},
    'ar': {'name': 'Arabic',   'family': 'Afro-Asiatic',  'color': '#42d4f4'},
    'fi': {'name': 'Finnish',  'family': 'Uralic',        'color': '#f032e6'},
}

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path("/content/drive/MyDrive/LRTIA/Data/wiki_multilingual")
    BASE_DIR = Path("/content/drive/MyDrive/LRTIA/Results/Wiki_multilingual_finegrain")
    DRIVE_RAID_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_finegrain")
    DRIVE_BUCKEYE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/Buckeye_finegrain")
    DRIVE_FRENCH_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/French_oral_finegrain")
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    DATA_DIR = Path("../data/wiki_multilingual")
    BASE_DIR = Path("../results/wiki_multilingual_finegrain")
    DRIVE_RAID_RESULTS = Path("../results/raid_finegrain")
    DRIVE_BUCKEYE_RESULTS = Path("../results/Buckeye_finegrain")
    DRIVE_FRENCH_RESULTS = Path("../results/French_oral_finegrain")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True
MAX_CONTEXT = 100
TARGET_LEN = 30
TARGET_FRACTIONS = [0.25, 0.50, 0.75]
MIN_CONTEXT_BEFORE_TARGET = MAX_CONTEXT + 10
RANDOM_SEED = 42

# Check which language files are available
available_langs = []
for code in LANGUAGES:
    path = DATA_DIR / f"{code}_articles.jsonl"
    if path.exists():
        available_langs.append(code)
    else:
        print(f"  WARNING: {path} not found, skipping {LANGUAGES[code]['name']}")

print(f"Languages available: {', '.join(LANGUAGES[code]['name'] for code in available_langs)}")
print(f"Max context: {MAX_CONTEXT} tokens, Target length: {TARGET_LEN} tokens")

In [ ]:
# === Load model ===
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )
model.eval()
print("Model loaded")

In [ ]:
# === Core functions ===

common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

@torch.no_grad()
def compute_ppl(token_ids, target_start, target_end):
    if target_start >= target_end - 1:
        return float('inf')
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')


def compute_token_reveal_curve(full_ids, target_start, target_end,
                                shuffled=False, rng_shuf=None):
    target_ids = full_ids[target_start:target_end]
    context_pool = list(full_ids[:target_start])
    if shuffled and rng_shuf is not None:
        context_pool = list(context_pool)
        rng_shuf.shuffle(context_pool)
    max_ctx = min(MAX_CONTEXT, len(context_pool))
    if max_ctx < 10:
        return None
    ppls, ctx_lengths = [], []
    for ctx_len in range(1, max_ctx + 1):
        ctx_tokens = context_pool[-ctx_len:]
        chunk = ctx_tokens + target_ids
        ppl = compute_ppl(chunk, len(ctx_tokens), len(chunk))
        if not math.isinf(ppl):
            ppls.append(ppl)
            ctx_lengths.append(ctx_len)
    if len(ppls) < 10:
        return None
    return {'ctx_lengths': ctx_lengths, 'ppls': ppls}


def process_document(doc, rng_shuf):
    full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
    n = len(full_ids)
    intact_curves, shuffled_curves = [], []
    for frac in TARGET_FRACTIONS:
        target_start = int(n * frac)
        target_end = min(target_start + TARGET_LEN, n)
        if target_start < MIN_CONTEXT_BEFORE_TARGET or target_end - target_start < 5:
            continue
        result = compute_token_reveal_curve(full_ids, target_start, target_end)
        if result is not None:
            result['doc_id'] = doc['doc_id']
            result['target_frac'] = frac
            intact_curves.append(result)
        result_s = compute_token_reveal_curve(full_ids, target_start, target_end,
                                              shuffled=True, rng_shuf=rng_shuf)
        if result_s is not None:
            result_s['doc_id'] = doc['doc_id']
            result_s['target_frac'] = frac
            shuffled_curves.append(result_s)
    return intact_curves, shuffled_curves


def compute_raw_ppl_curve(curves):
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    return np.nanmean(np.array(all_ppl), axis=0)


def fit_power_law(marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return slope, r, p, bc, bm, intercept
    return None

print("Functions defined")

In [ ]:
# === Process all languages ===
all_lang_results = {}  # lang_code -> {'intact': [...], 'shuffled': [...]}

for lang_code in available_langs:
    lang_info = LANGUAGES[lang_code]
    print(f"\n{'='*60}")
    print(f"Processing {lang_info['name']} ({lang_code}, {lang_info['family']})")
    print(f"{'='*60}")

    # Check for cached results
    intact_path = BASE_DIR / f"{lang_code}_intact_v1.json"
    shuffled_path = BASE_DIR / f"{lang_code}_shuffled_v1.json"

    if intact_path.exists() and shuffled_path.exists():
        with open(intact_path) as f:
            intact = json.load(f)
        with open(shuffled_path) as f:
            shuffled = json.load(f)
        print(f"  Loaded {len(intact)} intact + {len(shuffled)} shuffled curves from cache")
    else:
        # Load corpus
        corpus = []
        with open(DATA_DIR / f"{lang_code}_articles.jsonl") as f:
            for line in f:
                corpus.append(json.loads(line))
        print(f"  Loaded {len(corpus)} articles")

        # Process
        intact, shuffled = [], []
        rng_shuf = np.random.RandomState(RANDOM_SEED + 99)
        for doc in tqdm(corpus, desc=f"{lang_info['name']}"):
            i, s = process_document(doc, rng_shuf)
            intact.extend(i)
            shuffled.extend(s)

        # Cache
        with open(intact_path, 'w') as f:
            json.dump(intact, f)
        with open(shuffled_path, 'w') as f:
            json.dump(shuffled, f)
        print(f"  Computed {len(intact)} intact + {len(shuffled)} shuffled curves")

    all_lang_results[lang_code] = {'intact': intact, 'shuffled': shuffled}

    # Quick fit
    if len(intact) > 5 and len(shuffled) > 5:
        ip = compute_raw_ppl_curve(intact)
        sp = compute_raw_ppl_curve(shuffled)
        corr = -np.diff(ip) - (-np.diff(sp))
        fit = fit_power_law(corr)
        if fit:
            print(f"  >> {lang_info['name']} (corrected): alpha = {fit[0]:.3f} (r={fit[1]:.3f}, p={fit[2]:.4f})")
        else:
            print(f"  >> Power law fit failed")
    else:
        print(f"  >> Not enough curves for fit")

print("\n\nAll languages processed!")

In [ ]:
# === Compute all corrected marginals and fits ===
lang_fits = {}  # lang_code -> {slope, r, p, bc, bm, intercept, corrected_marg}

for lang_code, results in all_lang_results.items():
    intact = results['intact']
    shuffled = results['shuffled']
    if len(intact) < 5 or len(shuffled) < 5:
        continue
    ip = compute_raw_ppl_curve(intact)
    sp = compute_raw_ppl_curve(shuffled)
    corr = -np.diff(ip) - (-np.diff(sp))
    fit = fit_power_law(corr)
    if fit:
        lang_fits[lang_code] = {
            'slope': fit[0], 'r': fit[1], 'p': fit[2],
            'bc': fit[3], 'bm': fit[4], 'intercept': fit[5],
            'corrected_marg': corr,
            'intact_ppl': ip, 'shuffled_ppl': sp,
            'n_curves': len(intact),
        }

# Print summary table
print(f"\n{'Language':<12} {'Family':<16} {'α':>8} {'r':>8} {'p':>10} {'N curves':>10}")
print("-" * 68)
for code in available_langs:
    if code in lang_fits:
        f = lang_fits[code]
        info = LANGUAGES[code]
        print(f"{info['name']:<12} {info['family']:<16} {f['slope']:>8.3f} {f['r']:>8.3f} {f['p']:>10.4f} {f['n_curves']:>10}")
    else:
        print(f"{LANGUAGES[code]['name']:<12} — fit failed")

In [ ]:
# === Figure 1: Per-language panels (raw PPL + power law fit) ===
n_langs = len(lang_fits)
fig, axes = plt.subplots(2, n_langs, figsize=(5 * n_langs, 10))
if n_langs == 1:
    axes = axes.reshape(2, 1)

for i, (code, f) in enumerate(lang_fits.items()):
    info = LANGUAGES[code]
    color = info['color']

    # Top row: raw perplexity intact vs shuffled
    ax = axes[0, i]
    ax.plot(common_x, f['intact_ppl'], '-', color=color, linewidth=2, label='Intact')
    ax.plot(common_x, f['shuffled_ppl'], ':', color=color, linewidth=2, label='Shuffled')
    ax.set_xlabel('Context Length (tokens)')
    if i == 0: ax.set_ylabel('Perplexity')
    ax.set_title(f"{info['name']}\n({info['family']})", fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)

    # Bottom row: power law fit
    ax = axes[1, i]
    ax.plot(f['bc'], f['bm'], 'o-', color=color, linewidth=2, markersize=5)
    fit_x = np.linspace(min(f['bc']), max(f['bc']), 100)
    fit_y = np.exp(f['intercept']) * fit_x ** f['slope']
    ax.plot(fit_x, fit_y, '--', color=color, alpha=0.5,
            label=f"α={f['slope']:.2f} (r={f['r']:.2f})")
    ax.set_xscale('log')
    ax.set_xlabel('Distance (tokens, log)')
    if i == 0: ax.set_ylabel('Corrected Marginal')
    ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.2)

plt.suptitle('Coherence Decay Across 6 Non-Indo-European Languages',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_per_language.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Figure 2: Grand comparison — ALL languages + prior results ===

# Load prior results
prior_fits = []  # (label, exponent, r, color)

# RAID
raid_ip = DRIVE_RAID_RESULTS / "finegrain_results_v4.json"
raid_sp = DRIVE_RAID_RESULTS / "finegrain_shuffled_v4.json"
if raid_ip.exists() and raid_sp.exists():
    with open(raid_ip) as f: raid_all = json.load(f)
    with open(raid_sp) as f: raid_all_s = json.load(f)
    for pop, label, color in [('human', 'English written\n(RAID)', 'blue'),
                               ('ai', 'English AI\n(RAID)', 'red')]:
        ri = [c for c in raid_all if c['population'] == pop]
        rs = [c for c in raid_all_s if c['population'] == pop]
        rc = -np.diff(compute_raw_ppl_curve(ri)) - (-np.diff(compute_raw_ppl_curve(rs)))
        rf = fit_power_law(rc)
        if rf:
            prior_fits.append((label, rf[0], rf[1], color))

# Buckeye
bk_ip = DRIVE_BUCKEYE_RESULTS / "buckeye_intact_v1.json"
bk_sp = DRIVE_BUCKEYE_RESULTS / "buckeye_shuffled_v1.json"
if bk_ip.exists() and bk_sp.exists():
    with open(bk_ip) as f: bk_i = json.load(f)
    with open(bk_sp) as f: bk_s = json.load(f)
    bc = -np.diff(compute_raw_ppl_curve(bk_i)) - (-np.diff(compute_raw_ppl_curve(bk_s)))
    bf = fit_power_law(bc)
    if bf:
        prior_fits.append(('English spoken\n(Buckeye)', bf[0], bf[1], 'green'))

# French
fr_ip = DRIVE_FRENCH_RESULTS / "french_oral_intact_v1.json"
fr_sp = DRIVE_FRENCH_RESULTS / "french_oral_shuffled_v1.json"
if fr_ip.exists() and fr_sp.exists():
    with open(fr_ip) as f: fr_i = json.load(f)
    with open(fr_sp) as f: fr_s = json.load(f)
    fc = -np.diff(compute_raw_ppl_curve(fr_i)) - (-np.diff(compute_raw_ppl_curve(fr_s)))
    ff = fit_power_law(fc)
    if ff:
        prior_fits.append(('French spoken\n(Oral Narr.)', ff[0], ff[1], '#8B008B'))

print(f"Loaded {len(prior_fits)} prior results")
for label, exp, r, _ in prior_fits:
    print(f"  {label.replace(chr(10), ' ')}: α={exp:.3f} (r={r:.3f})")

In [ ]:
# === Figure 2: Grand bar chart — all exponents ===
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Collect all entries: (label, exponent, r, color, is_human)
entries = []

# New multilingual results
for code in available_langs:
    if code in lang_fits:
        f = lang_fits[code]
        info = LANGUAGES[code]
        entries.append((f"{info['name']}\n({info['family']})", f['slope'], f['r'], info['color'], True))

# Prior results
for label, exp, r, color in prior_fits:
    is_human = 'AI' not in label
    entries.append((label, exp, r, color, is_human))

# Anderson & Schooler reference
entries.append(('Anderson &\nSchooler', -0.77, None, 'gray', True))

# --- Panel A: Bar chart ---
ax = axes[0]
labels = [e[0] for e in entries]
exponents = [e[1] for e in entries]
colors = [e[3] for e in entries]

bars = ax.bar(range(len(entries)), exponents, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(entries)))
ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
ax.set_ylabel('Power Law Exponent (α)', fontsize=12)
ax.set_title('Coherence Decay Exponents Across Languages', fontweight='bold', fontsize=13)
ax.axhline(-0.77, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.grid(True, alpha=0.2, axis='y')

# Annotate
for i, (label, exp, r, color, is_human) in enumerate(entries):
    txt = f'{exp:.2f}'
    if r is not None: txt += f'\n(r={r:.2f})'
    y_offset = -0.06 if exp < 0 else 0.02
    ax.text(i, exp + y_offset, txt, ha='center', fontsize=7, fontweight='bold')

# --- Panel B: Human exponents only (excluding AI), sorted ---
ax = axes[1]
human_entries = [(l, e, r, c) for l, e, r, c, h in entries if h and r is not None]
human_entries.sort(key=lambda x: x[1])

h_labels = [e[0].replace('\n', ' ') for e in human_entries]
h_exps = [e[1] for e in human_entries]
h_colors = [e[3] for e in human_entries]

ax.barh(range(len(human_entries)), h_exps, color=h_colors, alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(human_entries)))
ax.set_yticklabels(h_labels, fontsize=9)
ax.set_xlabel('Power Law Exponent (α)', fontsize=12)
ax.set_title('Human Text Only (sorted)', fontweight='bold', fontsize=13)
ax.axvline(-0.77, color='gray', linestyle=':', alpha=0.5, label='Anderson & Schooler')

# Add mean line
mean_exp = np.mean(h_exps)
ax.axvline(mean_exp, color='black', linestyle='-', linewidth=2, alpha=0.5,
           label=f'Mean: {mean_exp:.2f}')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='x')

for i, exp in enumerate(h_exps):
    ax.text(exp - 0.03, i, f'{exp:.2f}', va='center', ha='right', fontsize=8, fontweight='bold')

plt.suptitle('Universal Coherence Decay: 8 Languages, 6 Language Families, Written + Spoken',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig2_grand_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print final table
print("\n" + "="*70)
print("GRAND SUMMARY")
print("="*70)
print(f"\n{'Condition':<35} {'α':>8} {'r':>8}")
print("-" * 55)
for label, exp, r, color, is_human in entries:
    r_str = f'{r:.3f}' if r is not None else '  —'
    marker = '  ' if is_human else '* '
    print(f"{marker}{label.replace(chr(10), ' '):<33} {exp:>8.3f} {r_str:>8}")
print(f"\nHuman mean (excl. AI): {mean_exp:.3f}")

In [ ]:
# === Figure 3: Overlaid corrected marginals for all languages ===
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

for code in available_langs:
    if code in lang_fits:
        f = lang_fits[code]
        info = LANGUAGES[code]
        ax.plot(common_x[1:], uniform_filter1d(f['corrected_marg'], 5),
                '-', color=info['color'], linewidth=2,
                label=f"{info['name']} ({info['family']}): α={f['slope']:.2f}")

# Add prior results as dashed
for label, exp, r, color in prior_fits:
    if 'AI' in label:
        continue  # skip AI for clarity
    # We'd need the marginals — just add reference lines instead

ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)', fontsize=13)
ax.set_ylabel('Corrected Marginal (intact - shuffled)', fontsize=13)
ax.set_title('Coherence Signal Across Non-Indo-European Languages', fontweight='bold', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(BASE_DIR / 'fig3_overlaid_marginals.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

If the exponents cluster near -0.75 across Chinese, Japanese, Korean, Turkish, Arabic, and Finnish — languages spanning 6 unrelated families with fundamentally different morphology, word order, and writing systems — then the coherence decay exponent is not just modality-invariant and language-family-invariant, but a **species-level universal** of human language production.

Combined with English written (-0.75), English spoken (-0.73), French spoken (-0.69), and Anderson & Schooler's memory retrieval statistics (-0.77), this would constitute strong evidence that the ~0.75 exponent reflects a **cognitive constraint** (working memory interference) rather than any property of specific languages or writing systems.